# Quickstart: container, MLflow, bucket directory, data in

- **Dataset:** the Pharma sample (`pharma_trials`, ClinicalTrials.gov studies) unless you pick another below: the
  Banking sample (`banking_complaints`), your own CSV, Parquet or JSON file, or a table in a connected source.
- **What must be loaded:** nothing beforehand. The sample-dataset cell loads it with `exakit data-load` the first
  time. Training needs a Python container in the database: `exakit udf-use default` (the `udf-devkit` add-on).
- **How long:** about five minutes, most of it loading the sample the first time.

First the three things training depends on, one cell each: each prints `ok` with what it found, or `FAIL` with the
reason and the fix. Then the same loader cells as the Data Scientist's quickstart.

## One workflow id on everything

The first cell starts a **workflow**: every step below - each check, each query, the role proposal and its approval -
appends one line carrying this id to `~/.exasol-starter-kit/workflows/<id>.jsonl`, and `01_train` tags its MLflow run
with it (`exakit workflow show <id>`). A kernel started with `EXAKIT_WORKFLOW_ID` set - the Data Scientist's, say -
continues that workflow instead of starting one.

In [ ]:
import os
import exakit.nb as nb

wid = nb.workflow("ml-engineer-quickstart", resume=os.environ.get("EXAKIT_WORKFLOW_ID"))
print(f"Every step below is logged under it: exakit workflow show {wid}")

## Which container trains

A training UDF runs on a Python container in the database, named by its **alias**. The `udf-devkit` add-on installs
two: `exakit udf-use default` puts the kit's pre-built one (scikit-learn, pandas and numpy pinned) in as
`PYTHON3_ML`; `exakit udf-setup` installs the official `PYTHON3`. This one assignment is the alias the container
check below and `01_train` use. Its default reads what the kit recorded as installed, `PYTHON3_ML` before `PYTHON3`
(`EXAKIT_UDF_CONTAINER` overrides it); with none installed it stops here and names the fix.

In [ ]:
CONTAINER = nb.ml_train.udf_container()   # or "PYTHON3_ML", "PYTHON3", or the alias of a container you installed
print(f"container alias: {CONTAINER}")

## Connected?

The kernel **Exasol (Starter Kit)** opens the database from the kit's encrypted store: there is no host, user or
password in this notebook. The second cell lists what the database holds; a virtual schema is a connected source.

In [ ]:
import exakit.nb as nb

conn = nb.connect()
nb.check_connection(conn)

In [ ]:
nb.list_schemas(conn)

## The container answers the smoke UDF

A small Python UDF runs on the container behind `CONTAINER` and reports Python and the packages the training UDF
needs. A `FAIL` line names the alias it probed and the command that installs that very container.

In [ ]:
container_ok = nb.check_container(conn, alias=CONTAINER)

## The MLflow server is reachable

`exakit mlflow start` runs the kit's local server; `exakit mlflow url` shows its address. Set
`MLFLOW_TRACKING_URI` before `exakit notebook` to use another server instead. If neither is running, runs still
go to the kit's local store under `~/.exasol-starter-kit/mlflow/`.

In [ ]:
mlflow_ok = nb.check_mlflow()

## The bucket directory is where the kit expects it

On a local deployment the database's bucket is a folder on this computer, read by UDFs under `/buckets`. The kit
names it (`exakit model list --json`); this checks the folder is there and that a UDF sees it.

In [ ]:
bucket_ok = nb.check_bucket(conn)

## Bring data in: one cell for each way

Set up the ones you want; a cell you leave as it is says so and skips. The data used from here on is the first
that loaded, in this order: a connected source, your own file, the sample dataset.

**A sample dataset** the kit ships: `pharma_trials` (ClinicalTrials.gov studies) or `banking_complaints` (CFPB
consumer complaints). The cell loads it with `exakit data-load` the first time (a few minutes) and uses it as it is
after that. `None` skips it.

In [ ]:
SAMPLE = "pharma_trials"   # or "banking_complaints", or None
sample = nb.load_sample(conn, SAMPLE)

**Your own file**: CSV, Parquet or JSON (an array of records, or one record per line). Name the table it goes into.
The next cell profiles its columns and lets you choose their roles; you do not have to write a descriptor.

In [ ]:
OWN_FILE = "~/my-data/transactions.csv"   # .csv, .parquet or .json
OWN_TABLE = "MY_DATA.TRANSACTIONS"
own = nb.load_own_file(OWN_FILE, OWN_TABLE, conn=conn, missing_ok=True)

**A connected source**: a table in a virtual schema, an external database Exasol reads live. Name it as
SCHEMA.TABLE; the next cell profiles a sample and lets you review the roles. With no source connected, it skips.

In [ ]:
SOURCE = None   # e.g. "VS_SALES.ORDERS"
source = nb.load_source(conn, SOURCE)

The data used from here on. A source wins over your file, which wins over the sample:

In [ ]:
ds = nb.first(source, own, sample)

## Review and approve the roles

The kit profiles the chosen table and shows the evidence behind each proposed role: one dropdown per column,
labelled with its type, null share and distinct values or span, the runner-up for its role under it. Pick a role
for each column, then click **Approve descriptor**. The next cell waits for your approval; it does not run on a
draft (in Marimo it stops until the button is clicked, then runs on its own). In a plain Python session, call
`review.approve({"COLUMN": "role", ...})` instead. A headless run (`EXAKIT_NB_AUTO_APPROVE=1`) approves only a
proposal equal to the current approved version, writing nothing; any other proposal stays a draft and the run
says why (`=force` approves it anyway).

In [ ]:
review = nb.review_roles(conn, ds)

In [ ]:
ds = review.dataset()
nb.show_roles(ds)

## A first query

Rows and positive labels per category, built from the descriptor's roles - the label is what `01_train` learns to
predict. A role no column fits (the descriptor's `missing`, with the profiler's reason) is not guessed: without a
category the rows are counted per entity, without a label the label count is left out, and the cell says so.

In [ ]:
skipped = {m["role"]: m["reason"] for m in ds.missing}
for role, instead in (("category", "rows are counted per entity instead"),
                      ("label", "the label count is left out, and 01_train has nothing to predict until a label role is approved")):
    if role in skipped:
        print(f"No {role} role ({skipped[role]}): {instead}; the steps that need it later say so.")
BY = ds.col("category") if ds.has("category") else ds.col("entity")
LABEL_1 = f",\n       SUM({ds.col('label')}) AS LABEL_1" if ds.has("label") else ""
first_rows = nb.sql(conn, f"""
SELECT {BY} AS CATEGORY,
       COUNT(*) AS N{LABEL_1}
FROM {ds.fact_table}
GROUP BY {BY}
ORDER BY N DESC
LIMIT 10""")
nb.show_sql()
first_rows

## Next: `01_train.ipynb`

Open `01_train.ipynb` in this folder: features in SQL from these roles, a scikit-learn classifier trained inside the
database, and the run in your MLflow with its metrics, training SQL and model. Run the Data Scientist's
`01_explore.ipynb` first to write the handoff. Then set the one DATASET assignment in `01_train` to the dataset id
or your file's path.